# P1-E10.1  External-Dataset Compatibility and Protocol Lock

**CAUEEG development cohort → AHEPA and Dryad/P-ADIC external validation**

## Goal

Audit two independent external EEG datasets and freeze the protocol that will be used in P1-E10.2 and P1-E10.3. This notebook performs **no model fitting, prediction, threshold selection, feature selection, or outcome-guided preprocessing**.

It produces separate eligibility decisions for:

- **AHEPA/OpenNeuro ds004504**: confirmatory CN vs AD validation.
- **Dryad/P-ADIC (10.5061/dryad.8gtht76pw)**: confirmatory CN vs AD and, only if distinct labels are empirically available, MCI vs AD and CN–MCI–AD validation.

Run all cells in order. Continue only for datasets whose final status is `READY`. The overall expected status is:

> **P1-E10.1 PASSED — DUAL EXTERNAL COMPATIBILITY AND PROTOCOL LOCKED**

> **Correction note:** the Dryad participant count is derived only from numeric HDF5 EEG datasets with one 19-channel axis and sufficient samples. MATLAB metadata/reference-array lengths are never treated as participant counts.


## Context & Methods

### Prespecified rules

1. CAUEEG remains the sole development dataset.
2. External diagnostic labels are used only to define prespecified endpoints and report cohort composition—not to choose preprocessing, features, models, thresholds, or exclusions.
3. AHEPA FTD and Dryad depression/schizophrenia are excluded from confirmatory CAUEEG endpoint validation.
4. The frozen CAUEEG reference preprocessing path, channel set, feature indices, estimators and decision rules must remain unchanged.
5. External recordings are resampled to the locked CAUEEG rate only during later preprocessing; this audit does not alter source data.
6. Predictions in P1-E10.2/3 will be aggregated once per external participant.
7. AHEPA and Dryad results will be reported separately before any synthesis.
8. Dataset incompatibility produces `STOP` rather than an improvised correction.

### Key assumptions

- The AHEPA BIDS `participant_id` identifies a unique participant.
- A Dryad MATLAB cell/reference entry may be treated as a participant only when its structure is empirically consistent with one 19-channel EEG recording per entry.
- Repository descriptions are contextual evidence; downloaded files determine actual counts and structure.

## Setup

### 1. Paths, frozen parameters and dependencies

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, re, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- USER-EDITABLE WINDOWS PATHS ----------------
PAPER_ROOT = Path(r'D:\EEG_AD_Research\paper1_experiments\paper1_outputs')
AHEPA_ROOT = Path(r'D:\EEG_AD_Research\data\ds004504')
DRYAD_ROOT = Path(r'D:\EEG_AD_Research\data\dryad-data')

E06_ROOT = PAPER_ROOT/'p1_e06_1_automated_qa_preprocessing_pilot'
E08_ROOT = PAPER_ROOT/'p1_e08_1_leakage_free_nonspiking_benchmark'
E09_2_ROOT = PAPER_ROOT/'p1_e09_2_exact_age_matched_sensitivity'
OUTPUT_ROOT = PAPER_ROOT/'p1_e10_1_dual_external_compatibility_protocol_lock'
TABLES = OUTPUT_ROOT/'tables'; FIGURES = OUTPUT_ROOT/'figures'
for path in (OUTPUT_ROOT, TABLES, FIGURES): path.mkdir(parents=True, exist_ok=True)

SEED = 20260828
TARGET_CHANNELS = ['Fp1','Fp2','F3','F4','C3','C4','P3','P4','O1','O2',
                   'F7','F8','T3','T4','T5','T6','Fz','Cz','Pz']
EXPECTED_SOURCE_SFREQ = {'AHEPA': 500.0, 'DRYAD': 500.0}
MIN_DURATION_SECONDS = 60.0
RELEVANT_LABELS = {'CN','MCI','AD'}
EXCLUDED_LABELS = {'FTD','DEPRESSION','SCHIZOPHRENIA','OTHER','UNKNOWN'}

def sha256_file(path, chunk=1024*1024):
    digest=hashlib.sha256()
    with open(path,'rb') as handle:
        for block in iter(lambda: handle.read(chunk),b''): digest.update(block)
    return digest.hexdigest()

def quick_fingerprint(path, block=1024*1024):
    path=Path(path); digest=hashlib.sha256(); size=path.stat().st_size
    with open(path,'rb') as handle:
        digest.update(handle.read(block))
        if size>block: handle.seek(max(0,size-block)); digest.update(handle.read(block))
    digest.update(str(size).encode())
    return digest.hexdigest()

def save_table(frame, name):
    path=TABLES/f'{name}.csv'; frame.to_csv(path,index=False)
    try: frame.to_excel(TABLES/f'{name}.xlsx',index=False)
    except Exception as exc: warnings.warn(f'Excel export skipped ({exc}); CSV was saved.')
    return path

def norm_channel(value):
    text=str(value).strip().upper().replace('EEG','')
    text=re.sub(r'[-_ ]?(REF|LE|RE|AVG|AV)$','',text).strip(' -_')
    aliases={'FP1':'Fp1','FP2':'Fp2','F3':'F3','F4':'F4','C3':'C3','C4':'C4',
             'P3':'P3','P4':'P4','O1':'O1','O2':'O2','F7':'F7','F8':'F8',
             'T3':'T3','T4':'T4','T5':'T5','T6':'T6','T7':'T3','T8':'T4',
             'P7':'T5','P8':'T6','FZ':'Fz','CZ':'Cz','PZ':'Pz'}
    return aliases.get(text,text)

def norm_label(value):
    text=re.sub(r'[^A-Z0-9]+','',str(value).upper())
    if text in {'A','AD','ALZ','ALZHEIMER','ALZHEIMERS','ALZHEIMERSDISEASE'}: return 'AD'
    if text in {'C','CN','HC','CONTROL','CONTROLS','NORMAL','HEALTHYCONTROL'}: return 'CN'
    if text in {'M','MCI','MILDCOGNITIVEIMPAIRMENT'}: return 'MCI'
    if text in {'F','FTD','FRONTOTEMPORALDEMENTIA'}: return 'FTD'
    if 'SCHIZ' in text: return 'SCHIZOPHRENIA'
    if 'DEP' in text: return 'DEPRESSION'
    return 'UNKNOWN'

print('Experiment: P1-E10.1')
print('AHEPA:',AHEPA_ROOT)
print('Dryad:',DRYAD_ROOT)
print('Output:',OUTPUT_ROOT)

## Data

### 2. Authenticate the frozen CAUEEG decision chain

In [ ]:
upstream_candidates = {
 'P1-E06.1 run record': E06_ROOT/'run_record.json',
 'P1-E08.1 run record': E08_ROOT/'run_record.json',
 'P1-E09.2 run record': E09_2_ROOT/'run_record.json',
}
model_paths = {task:E08_ROOT/'models'/f'{task}_LOCKED_NONSPIKING_BENCHMARK.joblib'
               for task in ['three_class','normal_vs_ad','mci_vs_ad']}
upstream_rows=[]
for name,path in {**upstream_candidates,**{f'model {k}':v for k,v in model_paths.items()}}.items():
    exists=path.is_file()
    status=''
    if exists and path.suffix.lower()=='.json':
        try: status=json.loads(path.read_text(encoding='utf-8')).get('status','')
        except Exception as exc: status=f'UNREADABLE: {exc}'
    upstream_rows.append({'artifact':name,'path':str(path),'exists':exists,
                          'sha256':sha256_file(path) if exists else '', 'reported_status':status})
upstream=pd.DataFrame(upstream_rows)
save_table(upstream,'01_CAUEEG_upstream_authentication')
display(upstream)

required_upstream = upstream['exists'].all()
e09_status = upstream.loc[upstream.artifact.eq('P1-E09.2 run record'),'reported_status'].iloc[0] if required_upstream else ''
if not required_upstream:
    raise FileNotFoundError('Frozen CAUEEG inputs are missing. Correct only the path block; do not substitute models.')
if 'PASSED' not in e09_status:
    raise RuntimeError(f'P1-E09.2 is not authenticated as PASSED: {e09_status}')
print('FROZEN CAUEEG CHAIN AUTHENTICATED')

### 3. Inventory AHEPA and Dryad source files

In [ ]:
if not AHEPA_ROOT.is_dir(): raise FileNotFoundError(f'AHEPA_ROOT not found: {AHEPA_ROOT}')
if not DRYAD_ROOT.is_dir(): raise FileNotFoundError(f'DRYAD_ROOT not found: {DRYAD_ROOT}')

ahepa_signal_files=[]
for pattern in ('*_eeg.set','*_eeg.edf','*_eeg.vhdr','*_eeg.fif'):
    ahepa_signal_files.extend(AHEPA_ROOT.rglob(pattern))
ahepa_signal_files=sorted(set(ahepa_signal_files))
# Prefer raw BIDS files. Derivatives are never silently mixed with raw recordings.
raw_bids_files=[p for p in ahepa_signal_files if 'derivatives' not in {part.lower() for part in p.parts}]
if raw_bids_files: ahepa_signal_files=raw_bids_files
participants_files=sorted(AHEPA_ROOT.rglob('participants.tsv'),key=lambda p:(len(p.parts),str(p)))
if not participants_files: raise FileNotFoundError('AHEPA participants.tsv not found.')
AHEPA_PARTICIPANTS=participants_files[0]

dryad_mat_files=sorted(DRYAD_ROOT.rglob('*.mat'))
inventory=[]
for dataset,paths in [('AHEPA',ahepa_signal_files),('DRYAD',dryad_mat_files)]:
    for path in paths:
        inventory.append({'dataset':dataset,'file_name':path.name,'path':str(path),
                          'size_GB':path.stat().st_size/1e9,
                          'quick_fingerprint_sha256':quick_fingerprint(path)})
inventory=pd.DataFrame(inventory)
save_table(inventory,'02_external_source_file_inventory')
display(inventory.groupby('dataset').agg(files=('file_name','size'),size_GB=('size_GB','sum')).round(3))
print('AHEPA participants:',AHEPA_PARTICIPANTS)

## Results

### 4. Audit AHEPA participant labels and EEG headers

In [ ]:
participants=pd.read_csv(AHEPA_PARTICIPANTS,sep='\t',dtype=str)
id_candidates=[c for c in participants.columns if c.lower() in {'participant_id','subject_id','subject'}]
label_candidates=[c for c in participants.columns if c.lower() in {'group','diagnosis','diagnostic_group','label'}]
if not id_candidates: raise KeyError(f'No AHEPA participant-ID column: {participants.columns.tolist()}')
if not label_candidates: raise KeyError(f'No AHEPA diagnosis/group column: {participants.columns.tolist()}')
id_col=id_candidates[0]; label_col=label_candidates[0]
participants['participant_id']=participants[id_col].astype(str).str.strip()
participants['external_label']=participants[label_col].map(norm_label)

def subject_from_path(path):
    match=re.search(r'(sub-[A-Za-z0-9]+)',str(path))
    return match.group(1) if match else ''

files=pd.DataFrame({'eeg_path':[str(p) for p in ahepa_signal_files]})
files['participant_id']=files.eeg_path.map(subject_from_path)
if files.participant_id.eq('').any(): raise RuntimeError('Could not obtain BIDS participant IDs from all AHEPA paths.')

try:
    import mne
except ImportError as exc:
    raise ImportError('Install MNE in eeg_snn: conda install -c conda-forge mne') from exc

def read_header(path):
    path=Path(path); suffix=path.suffix.lower()
    readers={'.set':mne.io.read_raw_eeglab,'.edf':mne.io.read_raw_edf,
             '.vhdr':mne.io.read_raw_brainvision,'.fif':mne.io.read_raw_fif}
    raw=readers[suffix](path,preload=False,verbose='ERROR')
    mapped=[norm_channel(ch) for ch in raw.ch_names]
    present=[ch for ch in TARGET_CHANNELS if ch in mapped]
    return {'sfreq_Hz':float(raw.info['sfreq']),'duration_seconds':float(raw.n_times/raw.info['sfreq']),
            'n_source_channels':len(raw.ch_names),'mapped_target_channels':len(set(present)),
            'missing_target_channels':'|'.join(ch for ch in TARGET_CHANNELS if ch not in mapped),
            'duplicate_mapped_channels':len(mapped)-len(set(mapped)),
            'source_channels':'|'.join(raw.ch_names)}

header_rows=[]
for number,row in files.iterrows():
    try: info=read_header(row.eeg_path); error=''
    except Exception as exc:
        info={'sfreq_Hz':np.nan,'duration_seconds':np.nan,'n_source_channels':np.nan,
              'mapped_target_channels':0,'missing_target_channels':'HEADER_READ_FAILED',
              'duplicate_mapped_channels':np.nan,'source_channels':''}; error=repr(exc)
    header_rows.append({**row.to_dict(),**info,'header_error':error})
    if (number+1)%20==0: print(f'AHEPA headers inspected: {number+1}/{len(files)}')
ahepa_headers=pd.DataFrame(header_rows)
ahepa=participants.merge(ahepa_headers,on='participant_id',how='left',validate='one_to_many',indicator=True)
ahepa['confirmatory_endpoint_eligible']=ahepa.external_label.isin(['CN','AD'])
ahepa['technical_eligible']=(ahepa._merge.eq('both') & ahepa.header_error.eq('') &
                             ahepa.mapped_target_channels.eq(len(TARGET_CHANNELS)) &
                             ahepa.duration_seconds.ge(MIN_DURATION_SECONDS))
ahepa['external_validation_eligible']=ahepa.confirmatory_endpoint_eligible & ahepa.technical_eligible
save_table(ahepa_headers,'03_AHEPA_EEG_header_audit')
save_table(ahepa,'04_AHEPA_participant_eligibility_manifest')
display(ahepa.groupby('external_label',dropna=False).agg(participants=('participant_id','nunique'),
        eeg_files=('eeg_path','count'),technically_eligible=('technical_eligible','sum'),
        validation_eligible=('external_validation_eligible','sum')))

### 5. Audit Dryad MATLAB structures without loading the full dataset

In [ ]:
def label_from_filename(name):
    text=name.lower()
    if any(x in text for x in ['control','ctrl','healthy','hc_']): return 'CN'
    if 'mci' in text: return 'MCI'
    if any(x in text for x in ['alz','ad_']): return 'AD'
    if 'schiz' in text: return 'SCHIZOPHRENIA'
    if any(x in text for x in ['dep','mdd']): return 'DEPRESSION'
    return 'UNKNOWN'

def inspect_hdf5_eeg_objects(path, expected_channels=19, source_sfreq=500.0):
    """Inventory every HDF5 dataset and retain only signal-shaped numeric arrays.

    A participant candidate must be a 2-D numeric dataset with exactly one
    19-channel axis and at least MIN_DURATION_SECONDS of samples on the other
    axis. Reference/cell array lengths are metadata and are never counts.
    """
    import h5py
    objects=[]; signals=[]
    with h5py.File(path,'r') as handle:
        def visit(name,obj):
            if not isinstance(obj,h5py.Dataset): return
            shape=tuple(int(x) for x in obj.shape)
            is_reference=(h5py.check_dtype(ref=obj.dtype) is not None or obj.dtype.kind=='O')
            numeric=(obj.dtype.kind in 'biufc') and not is_reference
            objects.append({'object':name,'shape':str(shape),'dtype':str(obj.dtype),
                            'reference_array':is_reference,'numeric':numeric})
            if numeric and len(shape)==2 and shape.count(expected_channels)==1:
                sample_count=max(shape)
                if sample_count >= int(MIN_DURATION_SECONDS*source_sfreq):
                    signals.append({'hdf5_object':name,'signal_shape':str(shape),
                                    'sample_count':sample_count,
                                    'duration_seconds_at_500Hz':sample_count/source_sfreq,
                                    'channel_axis':shape.index(expected_channels)})
        handle.visititems(visit)
    return objects,signals

structure_rows=[]; signal_rows=[]; dryad_file_rows=[]
for path in dryad_mat_files:
    label=label_from_filename(path.name)
    try:
        objects,signals=inspect_hdf5_eeg_objects(path)
        fmt='MATLAB_v7.3_HDF5'; error=''
    except Exception as exc:
        objects=[]; signals=[]; fmt='UNREADABLE_OR_NOT_HDF5'; error=repr(exc)
    for item in objects:
        structure_rows.append({'file_name':path.name,'external_label':label,**item})
    for item in signals:
        signal_rows.append({'file_name':path.name,'path':str(path),'external_label':label,**item})
    dryad_file_rows.append({'file_name':path.name,'path':str(path),'external_label':label,
                            'mat_format':fmt,'empirical_eeg_objects':len(signals),
                            'signal_shapes':'|'.join(sorted(set(x['signal_shape'] for x in signals))),
                            'structure_error':error})

dryad_files=pd.DataFrame(dryad_file_rows)
dryad_structure=pd.DataFrame(structure_rows)
dryad_signals=pd.DataFrame(signal_rows)
save_table(dryad_files,'05_DRYAD_MAT_file_summary')
save_table(dryad_structure,'06_DRYAD_MAT_structure_inventory')
save_table(dryad_signals,'06b_DRYAD_empirical_EEG_object_inventory')
display(dryad_files[['file_name','external_label','mat_format','empirical_eeg_objects','signal_shapes','structure_error']])

relevant_signals=dryad_signals.loc[dryad_signals.external_label.isin(RELEVANT_LABELS)].copy()
relevant_signals=relevant_signals.sort_values(['external_label','file_name','hdf5_object']).reset_index(drop=True)
relevant_signals['within_class_index']=relevant_signals.groupby('external_label').cumcount()+1
relevant_signals['participant_id']=[f'dryad_{label.lower()}_{index:04d}'
    for label,index in zip(relevant_signals.external_label,relevant_signals.within_class_index)]
dryad_manifest=relevant_signals.rename(columns={'hdf5_object':'source_hdf5_object'}).copy()
dryad_manifest['identity_basis']='one unique 19-channel numeric HDF5 EEG object'
dryad_manifest['external_validation_eligible']=True
save_table(dryad_manifest,'07_DRYAD_provisional_participant_manifest')
display(dryad_manifest.groupby('external_label').participant_id.nunique().rename('empirical_EEG_objects'))

# Fail loudly if the parser recreates the known metadata-array error pattern.
for invalid_count in (129,56):
    if invalid_count in set(dryad_manifest.groupby('external_label').size()):
        raise AssertionError('Dryad count equals a known metadata/reference-array length; participant inference is invalid.')

### 6. Freeze endpoint availability and dataset-specific readiness

In [ ]:
ahepa_counts=ahepa.loc[ahepa.external_validation_eligible].groupby('external_label').participant_id.nunique().to_dict()
dryad_counts=(dryad_manifest.groupby('external_label').participant_id.nunique().to_dict()
              if len(dryad_manifest) else {})

endpoint_rows=[]
for dataset,counts in [('AHEPA',ahepa_counts),('DRYAD',dryad_counts)]:
    endpoint_rows.extend([
      {'dataset':dataset,'endpoint':'normal_vs_ad','available':counts.get('CN',0)>0 and counts.get('AD',0)>0,
       'class_counts':json.dumps({'CN':counts.get('CN',0),'AD':counts.get('AD',0)})},
      {'dataset':dataset,'endpoint':'mci_vs_ad','available':counts.get('MCI',0)>0 and counts.get('AD',0)>0,
       'class_counts':json.dumps({'MCI':counts.get('MCI',0),'AD':counts.get('AD',0)})},
      {'dataset':dataset,'endpoint':'three_class','available':all(counts.get(x,0)>0 for x in ['CN','MCI','AD']),
       'class_counts':json.dumps({x:counts.get(x,0) for x in ['CN','MCI','AD']})},
    ])
endpoints=pd.DataFrame(endpoint_rows)
save_table(endpoints,'08_external_endpoint_availability')
display(endpoints)

ahepa_checks=pd.DataFrame([
 {'check':'AHEPA participant IDs unique','pass':not participants.participant_id.duplicated().any(),'detail':participants.participant_id.nunique()},
 {'check':'AHEPA EEG files found','pass':len(ahepa_signal_files)>0,'detail':len(ahepa_signal_files)},
 {'check':'all AHEPA EEG headers readable','pass':ahepa_headers.header_error.eq('').all(),'detail':int(ahepa_headers.header_error.ne('').sum())},
 {'check':'AHEPA CN and AD available','pass':all(ahepa_counts.get(x,0)>0 for x in ['CN','AD']),'detail':ahepa_counts},
 {'check':'eligible AHEPA files contain locked channels','pass':ahepa.loc[ahepa.confirmatory_endpoint_eligible,'mapped_target_channels'].eq(len(TARGET_CHANNELS)).all(),'detail':len(TARGET_CHANNELS)},
 {'check':'AHEPA files meet minimum duration','pass':ahepa.loc[ahepa.confirmatory_endpoint_eligible,'duration_seconds'].ge(MIN_DURATION_SECONDS).all(),'detail':MIN_DURATION_SECONDS},
])

dryad_checks=pd.DataFrame([
 {'check':'Dryad MATLAB files found','pass':len(dryad_mat_files)>0,'detail':len(dryad_mat_files)},
 {'check':'all Dryad MATLAB structures readable','pass':dryad_files.structure_error.eq('').all(),'detail':int(dryad_files.structure_error.ne('').sum())},
 {'check':'Dryad EEG objects empirically identified','pass':len(relevant_signals)>0 and relevant_signals.sample_count.ge(MIN_DURATION_SECONDS*EXPECTED_SOURCE_SFREQ['DRYAD']).all(),'detail':dryad_counts},
 {'check':'Dryad CN and AD available','pass':all(dryad_counts.get(x,0)>0 for x in ['CN','AD']),'detail':dryad_counts},
 {'check':'Dryad generated IDs unique','pass':len(dryad_manifest)>0 and not dryad_manifest.participant_id.duplicated().any(),'detail':len(dryad_manifest)},
 {'check':'Dryad signal arrays have empirical 19-channel axis','pass':len(relevant_signals)>0 and relevant_signals.signal_shape.str.contains('19').all(),'detail':'all retained HDF5 objects have one 19-channel axis; documented source rate 500 Hz'},
])
ahepa_ready=bool(ahepa_checks['pass'].all())
dryad_ready=bool(dryad_checks['pass'].all())
readiness=pd.DataFrame([{'dataset':'AHEPA','ready':ahepa_ready,'status':'READY' if ahepa_ready else 'STOP'},
                        {'dataset':'DRYAD','ready':dryad_ready,'status':'READY' if dryad_ready else 'STOP'}])
save_table(ahepa_checks,'09_AHEPA_readiness_checks'); save_table(dryad_checks,'10_DRYAD_readiness_checks')
save_table(readiness,'11_dataset_readiness_decisions')
display(ahepa_checks); display(dryad_checks); display(readiness)

### 7. Publication-ready compatibility overview

In [ ]:
plot_rows=[]
for dataset,counts in [('AHEPA',ahepa_counts),('DRYAD',dryad_counts)]:
    for label,count in counts.items():
        if label in RELEVANT_LABELS: plot_rows.append({'dataset':dataset,'label':label,'participants':count})
plot_data=pd.DataFrame(plot_rows)
fig,ax=plt.subplots(figsize=(7.2,4.4))
if len(plot_data):
    pivot=plot_data.pivot(index='dataset',columns='label',values='participants').fillna(0)
    pivot=pivot.reindex(columns=['CN','MCI','AD'],fill_value=0)
    pivot.plot(kind='bar',ax=ax,color=['#4C78A8','#F2CF5B','#E45756'])
ax.set(title='External cohorts available after compatibility audit',xlabel='',ylabel='Participants')
ax.tick_params(axis='x',rotation=0); ax.legend(title='Diagnostic group',frameon=False)
fig.tight_layout(); fig.savefig(FIGURES/'Figure_01_external_cohort_composition.png',dpi=600,bbox_inches='tight')
fig.savefig(FIGURES/'Figure_01_external_cohort_composition.pdf',bbox_inches='tight'); plt.show()

## Checks

### 8. Lock the dual external-validation protocol

In [ ]:
protocol={
 'experiment':'P1-E10.1 dual external compatibility and protocol lock',
 'development_dataset':'CAUEEG only',
 'external_datasets':['AHEPA ds004504','Dryad P-ADIC 10.5061/dryad.8gtht76pw'],
 'primary_endpoint':'normal_vs_ad',
 'secondary_endpoints':['mci_vs_ad','three_class'],
 'secondary_endpoint_rule':'evaluate only where native distinct MCI labels are empirically available',
 'excluded_confirmatory_labels':sorted(EXCLUDED_LABELS),
 'target_channels':TARGET_CHANNELS,
 'minimum_recording_duration_seconds':MIN_DURATION_SECONDS,
 'external_model_rule':'load frozen CAUEEG feature indices, scaler, estimator and decision rule; no refit',
 'preprocessing_rule':'apply frozen P1-E06.1 reference path; no external-outcome-guided changes',
 'aggregation_rule':'one prediction per external participant after prespecified epoch aggregation',
 'reporting_rule':'report AHEPA and Dryad separately before cross-dataset synthesis',
 'ahepa_ready':ahepa_ready,'dryad_ready':dryad_ready,
 'official_CAUEEG_test_reused_for_selection':False,
}
(OUTPUT_ROOT/'p1_e10_1_LOCKED_EXTERNAL_VALIDATION_PROTOCOL.json').write_text(json.dumps(protocol,indent=2),encoding='utf-8')
protocol_sha=sha256_file(OUTPUT_ROOT/'p1_e10_1_LOCKED_EXTERNAL_VALIDATION_PROTOCOL.json')

final_checks=pd.DataFrame([
 {'check':'frozen CAUEEG chain authenticated','pass':required_upstream and 'PASSED' in e09_status,'detail':e09_status},
 {'check':'both external source roots exist','pass':AHEPA_ROOT.is_dir() and DRYAD_ROOT.is_dir(),'detail':f'{AHEPA_ROOT} | {DRYAD_ROOT}'},
 {'check':'AHEPA readiness decision recorded','pass':True,'detail':'READY' if ahepa_ready else 'STOP independently'},
 {'check':'Dryad readiness decision recorded','pass':True,'detail':'READY' if dryad_ready else 'STOP independently'},
 {'check':'Normal-vs-AD endpoint prespecified','pass':protocol['primary_endpoint']=='normal_vs_ad','detail':'primary external endpoint'},
 {'check':'nonmatching diagnoses excluded','pass':all(x in EXCLUDED_LABELS for x in ['FTD','DEPRESSION','SCHIZOPHRENIA']),'detail':sorted(EXCLUDED_LABELS)},
 {'check':'no external model fitting performed','pass':True,'detail':'compatibility audit only'},
 {'check':'no external predictions performed','pass':True,'detail':'P1-E10.2 and P1-E10.3 only'},
 {'check':'no threshold or feature selection performed','pass':True,'detail':'frozen CAUEEG objects mandated'},
 {'check':'dataset-specific reporting locked','pass':True,'detail':'no naïve pooling'},
 {'check':'protocol JSON written and hashed','pass':bool(protocol_sha),'detail':protocol_sha},
])
save_table(final_checks,'12_final_integrity_checks'); display(final_checks)

status=('P1-E10.1 PASSED — DUAL EXTERNAL COMPATIBILITY AND PROTOCOL LOCKED'
        if final_checks['pass'].all() else 'STOP — P1-E10.1 INTEGRITY CHECKS FAILED')
run_record={'experiment':'P1-E10.1 CAUEEG dual external-dataset compatibility and protocol lock',
 'timestamp_utc':pd.Timestamp.utcnow().isoformat(),'status':status,
 'dataset_readiness':readiness.set_index('dataset').status.to_dict(),
 'protocol_json':str(OUTPUT_ROOT/'p1_e10_1_LOCKED_EXTERNAL_VALIDATION_PROTOCOL.json'),
 'protocol_sha256':protocol_sha,'upstream_artifact_hashes':upstream.set_index('artifact').sha256.to_dict(),
 'model_training_performed':False,'external_predictions_performed':False,
 'python':sys.version,'platform':platform.platform()}
(OUTPUT_ROOT/'run_record.json').write_text(json.dumps(run_record,indent=2),encoding='utf-8')
print(json.dumps(run_record,indent=2))
if not final_checks['pass'].all(): raise AssertionError(status)

## Next Steps

After the overall P1-E10.1 PASS:

1. Read `11_dataset_readiness_decisions.csv`.
2. If AHEPA is `READY`, run **P1-E10.2** once using the frozen Normal-vs-AD CAUEEG model.
3. If Dryad is `READY`, run **P1-E10.3** once. Evaluate only endpoints marked available in `08_external_endpoint_availability.csv`.
4. Do not repair a `STOP` by changing thresholds after reviewing diagnostic labels. Correct only demonstrable file parsing or metadata mapping errors, document the correction, and rerun P1-E10.1.
